In [ ]:
import kagglehub
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Read the CSV file using pandas
df = pd.read_csv(f"{path}/Q1_data.csv")  # Load dataset into DataFrame


In [ ]:
# Display the first few rows of the dataset
df.head()


In [ ]:
# Display dataset information
df.info()


In [ ]:
# Show statistical summary of numerical columns
df.describe()


In [ ]:
# Plot the target distribution (Delivery_Time)
plt.figure(figsize=(8,5))
sns.histplot(df["Delivery_Time"], kde=True)

plt.title("Distribution of Delivery Time")
plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.show()


In [ ]:
# Drop Order_ID since it is an identifier and not useful for prediction
df = df.drop(columns=["Order_ID"])


In [ ]:
# Check missing values in each column
df.isnull().sum()
# Fill missing values in numerical columns with the mean
num_cols = df.select_dtypes(include=["int64", "float64"]).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].mean())
# Fill missing values in categorical columns with the mode
cat_cols = df.select_dtypes(include=["object"]).columns

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
df


In [ ]:
# 3) Remove duplicates if any exist
dup_count = df.duplicated().sum()
print("Number of duplicate rows:", dup_count)

df = df.drop_duplicates()

In [ ]:
# 4) One-Hot Encode categorical variables
df = pd.get_dummies(df, drop_first=True)


In [ ]:
# 5) Feature scaling using StandardScaler
# Separate features and target
X = df.drop(columns=["Delivery_Time"])
y = df["Delivery_Time"]

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(" Data is ready!")
print("X_scaled shape:", X_scaled.shape)
print("y shape:", y.shape)


In [ ]:
print("Note: Target imbalance is not applicable because Delivery_Time is continuous (regression).")

In [ ]:

# 1) Split the dataset into features (X) and target (y)
X = df.drop(columns=["Delivery_Time"])   # Features
y = df["Delivery_Time"]                  # Target (continuous -> regression)


In [ ]:
# 2) Use the correct split: KFold (for regression)
kf = KFold(n_splits=5, shuffle=True, random_state=42)  # 5-Fold CV with shuffling

mae_scores = []  # Store MAE for each fold

# 3) Train a RandomForest model + 4) Evaluate using MAE only
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # Split data into train/test for this fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Create the RandomForest regressor model
    model = RandomForestRegressor(
        n_estimators=200,      # Number of trees
        random_state=42,       # Reproducibility
        n_jobs=-1              # Use all CPU cores
    )

    # Train the model
    model.fit(X_train, y_train)

    # Predict on the test fold
    y_pred = model.predict(X_test)

    # Calculate MAE for this fold
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

    print(f"Fold {fold} MAE: {mae:.4f}")  # Print fold result

# 5) Print the averaged score across all folds
avg_mae = np.mean(mae_scores)
print("-" * 30)
print(f"Average MAE across all folds: {avg_mae:.4f}")

In [ ]:
# Get feature importances from the trained RandomForest model
feature_importance = model.feature_importances_

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": feature_importance
}).sort_values(by="Importance", ascending=False)

# Plot feature importance
plt.figure(figsize=(10,6))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.gca().invert_yaxis()  # Most important feature on top
plt.title("Feature Importance from RandomForest Model")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

In [ ]:
# Predict delivery time using the trained model
y_pred_all = model.predict(X)

# Plot histogram of predicted delivery times
plt.figure(figsize=(8,5))
plt.hist(y_pred_all, bins=30)

plt.title("Predicted Delivery Time Distribution")
plt.xlabel("Predicted Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
try:
    from catboost import CatBoostRegressor
except ImportError:
    # If running on Colab, install CatBoost then import again
    !pip -q install catboost
    from catboost import CatBoostRegressor

# 1) Split the dataset into features (X) and target (y)
X = df.drop(columns=["Delivery_Time"])   # Features
y = df["Delivery_Time"]                  # Target (regression)

# 2) Use KFold for regression
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []  # Store MAE per fold

for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # Split data for this fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # 3) Model 1: RandomForestRegressor
    rf = RandomForestRegressor(
        n_estimators=300,     # Number of trees
        random_state=42,
        n_jobs=-1
    )

    # 3) Model 2: CatBoostRegressor (silent training)
    cb = CatBoostRegressor(
        iterations=600,       # Number of boosting rounds
        learning_rate=0.05,
        depth=8,
        random_seed=42,
        verbose=0            # No training logs
    )

    # Train both models
    rf.fit(X_train, y_train)
    cb.fit(X_train, y_train)

    # Predict with both models
    pred_rf = rf.predict(X_test)
    pred_cb = cb.predict(X_test)

    # 4) Average predictions (simple ensemble)
    pred_ensemble = (pred_rf + pred_cb) / 2

    # 5) Evaluate using MAE only
    mae = mean_absolute_error(y_test, pred_ensemble)
    mae_scores.append(mae)

    print(f"Fold {fold} Ensemble MAE: {mae:.4f}")

# Print the averaged MAE across all folds
avg_mae = np.mean(mae_scores)
print("-" * 30)
print(f"Average Ensemble MAE across all folds: {avg_mae:.4f}")